# Object Detection ด้วย Transformers และ YOLO: ตรวจจับ Pothole บนถนน

Notebook นี้พาคุณรู้จัก **Object Detection** ผ่าน 3 เส้นทาง บนภารกิจเดียวกัน — ตรวจจับ **pothole (หลุมบนถนน)** จากภาพถนนจริง

- **Path A:** Ultralytics YOLO11 — real-time detector ที่นิยมที่สุด
- **Path B:** HuggingFace DETR — transformer-based detector ที่ไม่ต้องใช้ NMS
- **Path C:** Zero-shot detectors (OWLv2, Grounding DINO) — ตรวจจับโดยไม่ต้องฝึก

**Dataset:** `chitholian/annotated-potholes-dataset` จาก Kaggle (665 ภาพ, Pascal VOC annotations, single class)

**Compute:** Colab free-tier T4 — ใช้เวลารวมประมาณ 25–30 นาที

**ความรู้ที่ต้องมีก่อน:** Python และ PyTorch พื้นฐาน


---
## Section 0 — การติดตั้งและตั้งค่า / Setup

ใน section นี้เราจะ:
1. ติดตั้ง libraries ทั้งหมดด้วย version ที่ระบุ
2. ตรวจสอบ GPU
3. **ตรวจสอบ Kaggle credentials** (ถ้าไม่มีจะหยุดและแสดงวิธีตั้งค่า)
4. ตั้งค่า random seeds และสร้าง directories


In [ ]:
# @title ติดตั้ง libraries
!pip install -q "ultralytics>=8.3,<9" "transformers>=4.46" datasets torchmetrics supervision albumentations pycocotools kagglehub timm


In [ ]:
# @title ตรวจสอบ GPU
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "❌ ไม่พบ GPU\n"
        "กรุณาเปลี่ยน runtime: Runtime → Change runtime type → T4 GPU"
    )

print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ CUDA: {torch.version.cuda}")
print(f"✓ PyTorch: {torch.__version__}")


### ตรวจสอบ Kaggle credentials

Notebook นี้ดาวน์โหลด dataset จาก Kaggle ผ่าน `kagglehub` ซึ่งต้องการ credentials ก่อนใช้งาน

**วิธีตั้งค่า credentials บน Colab:**

1. ไปที่ [https://www.kaggle.com/settings](https://www.kaggle.com/settings) → กดปุ่ม **Create New API Token** → จะได้ไฟล์ `kaggle.json`
2. เปิดไฟล์ `kaggle.json` แล้วดูค่า `username` กับ `key`
3. บน Colab กดไอคอนรูปกุญแจ 🔑 ทางซ้าย → เพิ่ม secret 2 ตัว:
   - `KAGGLE_USERNAME` = ค่า username
   - `KAGGLE_KEY` = ค่า key
4. เปิด toggle **Notebook access** ทั้งสอง secret
5. รัน cell ด้านล่างต่อ


In [ ]:
# @title ตรวจสอบ Kaggle credentials
import os
import json
from pathlib import Path

# Define the expected path for kaggle.json
kaggle_json_path = Path("kaggle.json")

try:
    if not kaggle_json_path.exists():
        raise FileNotFoundError(f"kaggle.json not found at {kaggle_json_path}")

    with open(kaggle_json_path, "r") as f:
        kaggle_creds = json.load(f)

    os.environ["KAGGLE_USERNAME"] = kaggle_creds["username"]
    os.environ["KAGGLE_KEY"] = kaggle_creds["key"]
    print("✓ โหลด Kaggle credentials จาก ~/.kaggle/kaggle.json เรียบร้อย")

    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        raise ValueError("Kaggle credentials (username and key) not found within kaggle.json.")

except Exception as e:
    raise SystemExit(
        f"❌ ไม่พบ Kaggle credentials: {e}\n"
        "กรุณาตรวจสอบว่าไฟล์ kaggle.json มีอยู่จริงและมี 'username' กับ 'key' ครบถ้วน"
    )

print("✓ Kaggle credentials พร้อมใช้งาน")

In [ ]:
# @title ตั้งค่า global config
import random
import numpy as np
from pathlib import Path

# Configuration
FAST_MODE = False  # True = ข้าม fine-tuning ใช้แค่ pretrained
SEED = 42

# Seed all RNGs
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Directories
ROOT = Path.cwd()
RUNS_DIR = ROOT / "runs"
OUTPUTS_DIR = ROOT / "outputs"
RUNS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

print(f"✓ ROOT: {ROOT}")
print(f"✓ FAST_MODE: {FAST_MODE}")
print(f"✓ SEED: {SEED}")


---
## Section 1 — พื้นฐาน Object Detection ใน 5 นาที / Object Detection Basics

ก่อนเริ่มฝึก model เราต้องเข้าใจแนวคิดพื้นฐาน 4 อย่างก่อน:
1. รูปแบบ bounding box
2. Intersection over Union (IoU)
3. Non-Maximum Suppression (NMS)
4. mean Average Precision (mAP)


### 1.1 รูปแบบกล่องครอบ (Bounding box formats)

ใน object detection เราใช้ **กล่องครอบ (bounding box)** ระบุตำแหน่งวัตถุ มีรูปแบบหลัก 3 แบบ:

| Format | ความหมาย | ใช้โดย |
|---|---|---|
| `xyxy` | (x_min, y_min, x_max, y_max) | PyTorch, torchvision |
| `xywh` | (x_min, y_min, width, height) | COCO Dataset |
| `cxcywh` | (center_x, center_y, width, height) | YOLO (normalized 0–1) |

**Normalized** หมายถึง หารค่าด้วยความกว้าง/สูงของภาพ ทำให้ค่าอยู่ในช่วง [0, 1]


In [ ]:
# @title Helper: แปลงรูปแบบ bounding box
def xyxy_to_xywh(box):
    """(x1,y1,x2,y2) → (x,y,w,h)"""
    x1, y1, x2, y2 = box
    return [x1, y1, x2 - x1, y2 - y1]

def xywh_to_cxcywh(box, img_w, img_h):
    """(x,y,w,h) absolute → (cx,cy,w,h) normalized"""
    x, y, w, h = box
    return [(x + w/2) / img_w, (y + h/2) / img_h, w / img_w, h / img_h]

# ทดสอบกับกล่องตัวอย่าง
example_xyxy = [100, 200, 300, 400]
example_xywh = xyxy_to_xywh(example_xyxy)
example_norm = xywh_to_cxcywh(example_xywh, img_w=640, img_h=480)

print(f"xyxy:    {example_xyxy}")
print(f"xywh:    {example_xywh}")
print(f"cxcywh (normalized): {[round(v, 3) for v in example_norm]}")


### 1.2 Intersection over Union (IoU)

**IoU** บอกว่า predicted box และ ground truth box ซ้อนทับกันมากแค่ไหน:

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

- IoU = 1 → ทับกันสนิท
- IoU = 0 → ไม่ทับกันเลย
- IoU > 0.5 → ถือว่าทำนายถูกในมาตรฐานทั่วไป


In [ ]:
# @title Visualize IoU
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(6, 6))

gt_box = (100, 100, 300, 300)      # ground truth
pred_box = (180, 150, 380, 350)    # prediction

ax.add_patch(patches.Rectangle(
    gt_box[:2], gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
    linewidth=3, edgecolor="green", facecolor="none", label="Ground truth"
))
ax.add_patch(patches.Rectangle(
    pred_box[:2], pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
    linewidth=3, edgecolor="red", facecolor="none", label="Prediction"
))

# Intersection
ix1, iy1 = max(gt_box[0], pred_box[0]), max(gt_box[1], pred_box[1])
ix2, iy2 = min(gt_box[2], pred_box[2]), min(gt_box[3], pred_box[3])
ax.add_patch(patches.Rectangle(
    (ix1, iy1), ix2-ix1, iy2-iy1,
    facecolor="blue", alpha=0.3, label="Intersection"
))

ax.set_xlim(50, 420); ax.set_ylim(50, 400); ax.invert_yaxis()
ax.legend(loc="upper right"); ax.set_title("IoU = Intersection / Union")
plt.show()


In [ ]:
# @title คำนวณ IoU แบบ manual และเทียบกับ torchvision
from torchvision.ops import box_iou

# Manual
ix1, iy1 = max(gt_box[0], pred_box[0]), max(gt_box[1], pred_box[1])
ix2, iy2 = min(gt_box[2], pred_box[2]), min(gt_box[3], pred_box[3])
inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
gt_area = (gt_box[2]-gt_box[0]) * (gt_box[3]-gt_box[1])
pred_area = (pred_box[2]-pred_box[0]) * (pred_box[3]-pred_box[1])
union = gt_area + pred_area - inter
iou_manual = inter / union

# Torchvision
boxes_t = torch.tensor([gt_box, pred_box], dtype=torch.float32)
iou_torch = box_iou(boxes_t[:1], boxes_t[1:]).item()

print(f"IoU (manual):      {iou_manual:.4f}")
print(f"IoU (torchvision): {iou_torch:.4f}")
print(f"✓ ค่าตรงกัน: {abs(iou_manual - iou_torch) < 1e-6}")


### 1.3 Non-Maximum Suppression (NMS)

Detector มักทำนายวัตถุเดียวกันหลายกล่องซ้อน ๆ กัน **NMS** ใช้ลบกล่องซ้ำ โดย:

1. เรียงกล่องตาม confidence จากมากไปน้อย
2. เลือกกล่องที่ confidence สูงสุด เก็บไว้
3. ลบกล่องอื่นที่มี IoU > threshold กับกล่องที่เก็บไว้
4. ทำซ้ำจนหมด


In [ ]:
# @title Visualize NMS
from torchvision.ops import nms

# 4 กล่องสมมติ: 3 กล่องแรกซ้อนกัน 1 กล่องคนละวัตถุ
boxes = torch.tensor([
    [100, 100, 300, 300],
    [105, 110, 305, 305],
    [110, 115, 310, 310],
    [350, 350, 500, 500],
], dtype=torch.float32)
scores = torch.tensor([0.95, 0.90, 0.85, 0.80])

keep = nms(boxes, scores, iou_threshold=0.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, idx, title in [(axes[0], range(len(boxes)), "ก่อน NMS (4 กล่อง)"),
                        (axes[1], keep.tolist(),    f"หลัง NMS ({len(keep)} กล่อง)")]:
    for i in idx:
        x1, y1, x2, y2 = boxes[i].tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor="red", facecolor="none"))
        ax.text(x1, y1-5, f"{scores[i]:.2f}", color="red")
    ax.set_xlim(50, 550); ax.set_ylim(50, 550); ax.invert_yaxis()
    ax.set_title(title)
plt.show()

print(f"✓ NMS ลบกล่องซ้ำไป {len(boxes) - len(keep)} กล่อง")


### 1.4 mAP@0.5 vs mAP@0.5:0.95

**mAP (mean Average Precision)** เป็น metric หลักของ object detection:

- **mAP@0.5** — ใช้ IoU threshold = 0.5 ในการตัดสินว่ากล่องทำนายถูก (เกณฑ์หลวม)
- **mAP@0.5:0.95** — เฉลี่ย mAP จาก IoU threshold 0.5 ถึง 0.95 (เพิ่มทีละ 0.05) — เป็น COCO standard, เข้มงวดกว่า

mAP@0.5 จะมีค่าสูงกว่า mAP@0.5:0.95 เสมอ เพราะยอมรับกล่องที่ตำแหน่งคลาดเคลื่อนได้มากกว่า

(เราจะวัดค่าจริงใน Section 5)


### 1.5 ตระกูล detector

Detectors แบ่งเป็น 3 ตระกูลหลัก:

**1. Anchor-based** (YOLOv5, Faster R-CNN) — ใช้ anchor boxes ที่กำหนดล่วงหน้า ต้องปรับ anchor ให้เข้ากับ dataset

**2. Anchor-free** (YOLO11, FCOS) — ทำนายจุดศูนย์กลางวัตถุโดยตรง ไม่ต้องออกแบบ anchor

**3. Query-based** (DETR, RT-DETR, OWLv2) — ใช้ transformer attention DETR/RT-DETR ไม่ต้องใช้ NMS เลย OWLv2 และ Grounding DINO ทำ open-vocabulary detection ผ่าน text prompt ได้

ใน notebook นี้:
- Section 3 ใช้ YOLO11 (anchor-free)
- Section 4 ใช้ DETR (query-based)
- Section 6 ใช้ OWLv2 และ Grounding DINO (query-based + text-conditioned)


**สิ่งที่ได้เรียนรู้**
- bounding box มี 3 รูปแบบหลัก: xyxy, xywh, cxcywh
- IoU วัดความซ้อนทับ NMS ใช้ลบกล่องซ้ำ mAP เป็น metric หลัก
- detector มี 3 ตระกูล — anchor-based, anchor-free, query-based


---
## Section 2 — ชุดข้อมูล Pothole / Dataset

ใน section นี้เราจะ:
1. ดาวน์โหลด `chitholian/annotated-potholes-dataset` จาก Kaggle
2. parse Pascal VOC XML annotations
3. แบ่ง train/val/test (80/10/10) ด้วย seed คงที่
4. แปลงเป็น YOLO format (สำหรับ Path A) และ HuggingFace format (สำหรับ Path B) จาก source เดียวกัน
5. เลือก `TEST_IMAGES` 3 ภาพคงที่ ใช้ตลอดทั้ง notebook
6. visualize ตัวอย่าง


In [ ]:
# @title ดาวน์โหลด dataset
import kagglehub

print("กำลังดาวน์โหลด dataset (~50 MB)...")
KAGGLE_PATH = Path(kagglehub.dataset_download("chitholian/annotated-potholes-dataset"))
print(f"✓ ดาวน์โหลดเสร็จ: {KAGGLE_PATH}")


In [ ]:
# @title สำรวจโครงสร้าง dataset
def show_tree(root: Path, max_depth: int = 2, max_files: int = 5):
    """แสดง tree แบบจำกัดความลึกและจำนวนไฟล์"""
    for path in sorted(root.iterdir()):
        if path.is_dir():
            print(f"📁 {path.relative_to(root)}/")
            if max_depth > 1:
                files = list(path.iterdir())
                for child in files[:max_files]:
                    print(f"   {child.name}")
                if len(files) > max_files:
                    print(f"   ... อีก {len(files) - max_files} ไฟล์")
        else:
            print(f"📄 {path.relative_to(root)}")

show_tree(KAGGLE_PATH)


### Pascal VOC format

annotations อยู่ใน XML ตาม Pascal VOC format ตัวอย่างหนึ่งไฟล์มีโครงสร้างประมาณนี้:

```xml
<annotation>
  <size><width>720</width><height>540</height></size>
  <object>
    <name>pothole</name>
    <bndbox>
      <xmin>120</xmin><ymin>250</ymin>
      <xmax>340</xmax><ymax>410</ymax>
    </bndbox>
  </object>
  ...
</annotation>
```

เราต้อง parse XML แล้วแปลง `(xmin, ymin, xmax, ymax)` ไปเป็นรูปแบบที่ YOLO และ DETR ต้องการ


In [ ]:
# @title หา image-XML pairs ใน dataset
import xml.etree.ElementTree as ET

def find_pairs(root: Path):
    """เดิน directory tree หา .jpg/.png พร้อม .xml ที่ matching กัน"""
    img_exts = {".jpg", ".jpeg", ".png"}
    images = {p.stem: p for p in root.rglob("*") if p.suffix.lower() in img_exts}
    xmls = {p.stem: p for p in root.rglob("*.xml")}
    common = sorted(set(images) & set(xmls))
    return [(images[k], xmls[k]) for k in common]

pairs = find_pairs(KAGGLE_PATH)
print(f"✓ พบ image-XML pairs จำนวน {len(pairs)} คู่")
print(f"\nตัวอย่าง 3 คู่แรก:")
for img, xml in pairs[:3]:
    print(f"  {img.name}  ↔  {xml.name}")


In [ ]:
# @title Parser: Pascal VOC XML → unified record
def parse_voc(xml_path: Path):
    """อ่าน Pascal VOC XML คืน dict {width, height, boxes:[{xmin,ymin,xmax,ymax,name}]}"""
    root = ET.parse(xml_path).getroot()

    size = root.find("size")
    width = int(float(size.find("width").text))
    height = int(float(size.find("height").text))

    boxes = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        bb = obj.find("bndbox")
        # บางไฟล์อาจมีค่าเป็น float ใช้ float() แล้วแปลง int
        xmin = max(0, int(float(bb.find("xmin").text)))
        ymin = max(0, int(float(bb.find("ymin").text)))
        xmax = min(width,  int(float(bb.find("xmax").text)))
        ymax = min(height, int(float(bb.find("ymax").text)))
        if xmax > xmin and ymax > ymin:
            boxes.append({"name": name, "xmin": xmin, "ymin": ymin,
                          "xmax": xmax, "ymax": ymax})
    return {"width": width, "height": height, "boxes": boxes}

# Build records
records = []
for img_path, xml_path in pairs:
    parsed = parse_voc(xml_path)
    if parsed["boxes"]:  # ข้ามภาพที่ไม่มี bbox
        records.append({
            "image_path": img_path,
            "xml_path": xml_path,
            "width": parsed["width"],
            "height": parsed["height"],
            "boxes": parsed["boxes"],
        })

print(f"✓ Parse สำเร็จ {len(records)} ภาพ (ภาพที่มี bbox อย่างน้อย 1 อัน)")

# สถิติ
n_boxes = sum(len(r["boxes"]) for r in records)
class_names = sorted({b["name"] for r in records for b in r["boxes"]})
print(f"  จำนวน bounding boxes ทั้งหมด: {n_boxes}")
print(f"  Classes: {class_names}")


In [ ]:
# @title แบ่ง train / val / test แบบ seeded (80/10/10)
random.seed(SEED)
records_shuffled = records.copy()
random.shuffle(records_shuffled)

n = len(records_shuffled)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_records = records_shuffled[:n_train]
val_records   = records_shuffled[n_train:n_train + n_val]
test_records  = records_shuffled[n_train + n_val:]

print(f"✓ Split (seed={SEED}):")
print(f"  train: {len(train_records)} ภาพ")
print(f"  val:   {len(val_records)} ภาพ")
print(f"  test:  {len(test_records)} ภาพ")


### แปลงเป็น YOLO format

YOLO ต้องการ:
- `images/{split}/*.jpg`
- `labels/{split}/*.txt` — แต่ละบรรทัดคือ `class_id cx cy w h` (normalized 0–1)
- `data.yaml` ระบุ path, classes, จำนวน classes


In [ ]:
# @title เขียน YOLO format
import shutil

YOLO_ROOT = ROOT / "data_yolo"

def write_yolo_split(records_list, split: str):
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for rec in records_list:
        # คัดลอกภาพ
        dest_img = img_dir / rec["image_path"].name
        if not dest_img.exists():
            shutil.copy(rec["image_path"], dest_img)

        # เขียน label (normalized cxcywh, class 0 = pothole)
        w, h = rec["width"], rec["height"]
        lines = []
        for box in rec["boxes"]:
            cx = ((box["xmin"] + box["xmax"]) / 2) / w
            cy = ((box["ymin"] + box["ymax"]) / 2) / h
            bw = (box["xmax"] - box["xmin"]) / w
            bh = (box["ymax"] - box["ymin"]) / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        (lbl_dir / (rec["image_path"].stem + ".txt")).write_text("\n".join(lines))

write_yolo_split(train_records, "train")
write_yolo_split(val_records, "val")
write_yolo_split(test_records, "test")

# เขียน data.yaml
yaml_text = f"""path: {YOLO_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: pothole
"""
yaml_path = YOLO_ROOT / "data.yaml"
yaml_path.write_text(yaml_text)

print(f"✓ YOLO format เขียนที่ {YOLO_ROOT}")
print(f"  data.yaml: {yaml_path}")
print(f"\n--- data.yaml ---")
print(yaml_text)


### แปลงเป็น HuggingFace Dataset format

HF/DETR ต้องการ COCO-style bbox: `[x_min, y_min, width, height]` (absolute pixels)


In [ ]:
# @title สร้าง HuggingFace Dataset (COCO bbox format)
from datasets import Dataset, DatasetDict, Features, Value, Sequence, Image as HFImage

def records_to_hf(records_list):
    rows = []
    for i, rec in enumerate(records_list):
        boxes_coco = []
        for box in rec["boxes"]:
            x = box["xmin"]; y = box["ymin"]
            w = box["xmax"] - box["xmin"]
            h = box["ymax"] - box["ymin"]
            boxes_coco.append([float(x), float(y), float(w), float(h)])

        rows.append({
            "image": str(rec["image_path"]),
            "image_id": i,
            "width": rec["width"],
            "height": rec["height"],
            "objects": {
                "id":       list(range(len(boxes_coco))),
                "area":     [float(b[2] * b[3]) for b in boxes_coco],
                "bbox":     boxes_coco,
                "category": [0] * len(boxes_coco),
            },
        })

    features = Features({
        "image":    HFImage(),
        "image_id": Value("int64"),
        "width":    Value("int32"),
        "height":   Value("int32"),
        "objects":  Sequence({
            "id":       Value("int64"),
            "area":     Value("float32"),
            "bbox":     Sequence(Value("float32"), length=4),
            "category": Value("int64"),
        }),
    })
    return Dataset.from_list(rows, features=features)

hf_dataset = DatasetDict({
    "train":      records_to_hf(train_records),
    "validation": records_to_hf(val_records),
    "test":       records_to_hf(test_records),
})

print(f"✓ HF dataset:")
for split, ds in hf_dataset.items():
    print(f"  {split:11s}: {len(ds)} ตัวอย่าง")

# ตรวจสอบ split sync
assert len(hf_dataset["train"]) == len(train_records)
assert len(hf_dataset["validation"]) == len(val_records)
assert len(hf_dataset["test"]) == len(test_records)
print("✓ ยืนยัน YOLO และ HF ใช้ split เดียวกัน")


In [ ]:
# @title เลือก TEST_IMAGES 3 ภาพคงที่ — ใช้ตลอดทั้ง notebook
# จัด sort เพื่อ deterministic แล้วใช้ seed เลือก 3 ภาพแรกของ shuffled test set
test_imgs_sorted = sorted([r["image_path"] for r in test_records])
random.seed(SEED)
random.shuffle(test_imgs_sorted)
TEST_IMAGES = [str(p) for p in test_imgs_sorted[:3]]

print("✓ TEST_IMAGES (ใช้ใน Section 3, 4, 5, 6):")
for i, p in enumerate(TEST_IMAGES, 1):
    print(f"  {i}. {Path(p).name}")


In [ ]:
# @title visualize 6 ตัวอย่างจาก train set
import supervision as sv
import cv2

CLASS_NAMES = ["pothole"]
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

random.seed(SEED)
sample_idx = random.sample(range(len(hf_dataset["train"])), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, idx in zip(axes.flat, sample_idx):
    sample = hf_dataset["train"][idx]
    img = np.array(sample["image"])

    bboxes_coco = sample["objects"]["bbox"]
    if len(bboxes_coco):
        xyxy = np.array([[b[0], b[1], b[0]+b[2], b[1]+b[3]] for b in bboxes_coco])
        det = sv.Detections(
            xyxy=xyxy,
            class_id=np.array(sample["objects"]["category"]),
            confidence=np.ones(len(xyxy)),
        )
        labels = ["pothole"] * len(xyxy)
        annotated = box_annotator.annotate(img.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img

    ax.imshow(annotated)
    ax.set_title(f"id={sample['image_id']}, boxes={len(bboxes_coco)}")
    ax.axis("off")

plt.tight_layout(); plt.show()


In [ ]:
# @title Histogram ขนาด bounding box
all_areas = []
all_aspect = []
for rec in train_records:
    for b in rec["boxes"]:
        bw = b["xmax"] - b["xmin"]
        bh = b["ymax"] - b["ymin"]
        all_areas.append(bw * bh)
        all_aspect.append(bw / max(bh, 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(all_areas, bins=40, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].axvline(np.median(all_areas), color="red", linestyle="--",
                label=f"median = {np.median(all_areas):.0f} px²")
axes[0].set_xlabel("Bbox area (px²)"); axes[0].set_ylabel("count")
axes[0].set_title("Distribution of bbox area"); axes[0].legend()

axes[1].hist(all_aspect, bins=40, color="seagreen", edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Aspect ratio (w/h)"); axes[1].set_ylabel("count")
axes[1].set_title("Distribution of aspect ratio")
plt.tight_layout(); plt.show()

print(f"จำนวน bbox ใน train: {len(all_areas)}")
print(f"Median area: {np.median(all_areas):.0f} px² → imgsz=640 น่าจะเพียงพอ")


**สิ่งที่ได้เรียนรู้**
- Pascal VOC ใช้ XML เก็บ `(xmin, ymin, xmax, ymax)` — ต้อง parse และแปลงให้เข้ากับ YOLO/COCO format
- เราสร้าง YOLO และ HF จาก source records ชุดเดียว ทำให้ split sync กันแน่นอน
- `TEST_IMAGES` คงที่ ใช้ทุก section เพื่อเปรียบเทียบผลได้ตรง ๆ


---
## Section 3 — Path A: Ultralytics YOLO11

ใน section นี้:
1. ใช้ pretrained YOLO11n inference บน TEST_IMAGES
2. Fine-tune YOLO11n บน pothole dataset
3. ดู training curves และ mAP
4. เทียบ pretrained vs fine-tuned แบบ side-by-side
5. Export เป็น ONNX

**Note สำคัญ:** YOLO11n pretrained บน COCO ซึ่ง**ไม่มี class "pothole"** ดังนั้น pretrained model จะ**ตรวจไม่เจอ pothole เลย** ส่วนที่มันตรวจเจอคือ class อื่น (เช่น `car`) ที่บังเอิญอยู่ในภาพ — ปรากฏการณ์นี้เป็นเหตุผลที่เราต้อง fine-tune


### 3a. Pretrained YOLO11n inference

In [ ]:
# @title Load pretrained YOLO11n + inference บน TEST_IMAGES
from ultralytics import YOLO

model_yolo_pre = YOLO("yolo11n.pt")
COCO_NAMES = model_yolo_pre.names  # dict {0: 'person', 1: 'bicycle', ...}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img_path in zip(axes, TEST_IMAGES):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    results = model_yolo_pre(img_path, verbose=False, conf=0.25)

    boxes_xyxy = []
    confs = []
    cls_ids = []
    for r in results:
        for b in r.boxes:
            boxes_xyxy.append(b.xyxy[0].cpu().numpy())
            confs.append(float(b.conf[0]))
            cls_ids.append(int(b.cls[0]))

    if boxes_xyxy:
        det = sv.Detections(
            xyxy=np.array(boxes_xyxy),
            confidence=np.array(confs),
            class_id=np.array(cls_ids),
        )
        labels = [f"{COCO_NAMES[c]} {conf:.2f}" for c, conf in zip(cls_ids, confs)]
        annotated = box_annotator.annotate(img.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img

    ax.imshow(annotated)
    ax.set_title(f"Pretrained YOLO11n\n{Path(img_path).name}")
    ax.axis("off")

plt.tight_layout(); plt.show()

print("\n📌 สังเกต: pretrained YOLO11n ไม่ตรวจเจอ pothole เลย")
print("   เพราะ COCO 80 classes ไม่มี 'pothole' — model ไม่เคยเห็นมา")
print(f"   COCO classes: {list(COCO_NAMES.values())[:10]}, ... (ทั้งหมด {len(COCO_NAMES)} classes)")


### 3b. Fine-tune YOLO11n บน pothole dataset

**Hyperparameters ที่ปรับให้พอดีกับ Colab T4:**
- `epochs=15` — รวมประมาณ 5–7 นาที
- `imgsz=640` — มาตรฐาน
- `batch=16` — fit ใน 16 GB VRAM ของ T4
- `cache=True` — แคช dataset ใน RAM
- `patience=5` — early stop ถ้า val mAP ไม่ขึ้น 5 epochs


In [ ]:
# @title Fine-tune YOLO11n
import time

if FAST_MODE:
    print("⚠️ FAST_MODE = True: ข้าม fine-tuning")
    model_yolo_ft = model_yolo_pre  # fallback to pretrained
else:
    model_yolo = YOLO("yolo11n.pt")

    t0 = time.time()
    results = model_yolo.train(
        data=str(yaml_path),
        epochs=15,
        imgsz=640,
        batch=16,
        cache=True,
        patience=5,
        plots=True,
        verbose=False,
        project=str(RUNS_DIR),
        name="yolo11n_pothole",
        exist_ok=True,
        seed=SEED,
    )
    yolo_train_time = time.time() - t0

    best_pt = Path(results.save_dir) / "weights" / "best.pt"
    model_yolo_ft = YOLO(str(best_pt))
    print(f"\n✓ ฝึกเสร็จใน {yolo_train_time/60:.1f} นาที")
    print(f"✓ best weights: {best_pt}")


In [ ]:
# @title แสดง training curves
from PIL import Image as PILImage

if FAST_MODE:
    print("⚠️ FAST_MODE: ไม่มี curves")
else:
    results_png = Path(model_yolo_ft.ckpt_path).parent.parent / "results.png"
    if results_png.exists():
        plt.figure(figsize=(14, 8))
        plt.imshow(PILImage.open(results_png))
        plt.axis("off")
        plt.title("YOLO11n training curves")
        plt.show()
    else:
        print(f"⚠️ ไม่พบไฟล์ results.png ที่ {results_png}")


In [ ]:
# @title ประเมินด้วย model.val()
if FAST_MODE:
    yolo_metrics = {"map50": 0.0, "map": 0.0, "precision": 0.0, "recall": 0.0}
    print("⚠️ FAST_MODE: ใช้ค่า dummy")
else:
    val_results = model_yolo_ft.val(data=str(yaml_path), verbose=False, plots=False)
    yolo_metrics = {
        "map50":     float(val_results.box.map50),
        "map":       float(val_results.box.map),
        "precision": float(val_results.box.mp),
        "recall":    float(val_results.box.mr),
    }
    print("📊 Fine-tuned YOLO11n บน validation set:")
    print(f"  mAP@0.5      : {yolo_metrics['map50']:.4f}")
    print(f"  mAP@0.5:0.95 : {yolo_metrics['map']:.4f}")
    print(f"  Precision    : {yolo_metrics['precision']:.4f}")
    print(f"  Recall       : {yolo_metrics['recall']:.4f}")


In [ ]:
# @title เปรียบเทียบ pretrained vs fine-tuned บน TEST_IMAGES
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col, img_path in enumerate(TEST_IMAGES):
    img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    for row, (model_inst, row_label) in enumerate([
        (model_yolo_pre, "Pretrained (COCO)"),
        (model_yolo_ft,  "Fine-tuned (pothole)"),
    ]):
        results = model_inst(img_path, verbose=False, conf=0.25)
        boxes_xyxy, confs, cls_ids = [], [], []
        for r in results:
            for b in r.boxes:
                boxes_xyxy.append(b.xyxy[0].cpu().numpy())
                confs.append(float(b.conf[0]))
                cls_ids.append(int(b.cls[0]))

        if boxes_xyxy:
            det = sv.Detections(
                xyxy=np.array(boxes_xyxy),
                confidence=np.array(confs),
                class_id=np.array(cls_ids),
            )
            names = model_inst.names
            labels = [f"{names[c]} {conf:.2f}" for c, conf in zip(cls_ids, confs)]
            annotated = box_annotator.annotate(img_rgb.copy(), det)
            annotated = label_annotator.annotate(annotated, det, labels)
        else:
            annotated = img_rgb

        axes[row, col].imshow(annotated)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(row_label, fontsize=14, fontweight="bold")
        axes[row, col].set_title(Path(img_path).name)

# Add row labels via figure text
fig.text(0.02, 0.75, "Pretrained\n(COCO)", fontsize=14, fontweight="bold",
         rotation=90, va="center", ha="center")
fig.text(0.02, 0.27, "Fine-tuned\n(pothole)", fontsize=14, fontweight="bold",
         rotation=90, va="center", ha="center")

plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.show()


In [ ]:
# @title Export → ONNX
yolo_onnx_path = OUTPUTS_DIR / "yolo11n_pothole.onnx"

if FAST_MODE:
    print("⚠️ FAST_MODE: ข้าม ONNX export")
else:
    print("กำลัง export → ONNX...")
    onnx_src = Path(model_yolo_ft.export(format="onnx", dynamic=True))
    shutil.copy(onnx_src, yolo_onnx_path)
    size_mb = yolo_onnx_path.stat().st_size / 1e6
    print(f"✓ ONNX: {yolo_onnx_path} ({size_mb:.2f} MB)")
    print("\nหมายเหตุ: รูปแบบอื่นที่ export ได้ — TensorRT (`engine`), CoreML, TorchScript")


**สิ่งที่ได้เรียนรู้**
- pretrained YOLO11n ตรวจไม่เจอ pothole เพราะ COCO ไม่มี class นี้
- fine-tune ด้วย epochs น้อย ๆ บน T4 ก็ทำให้ mAP สูงขึ้นมาก
- export → ONNX ทำได้ในบรรทัดเดียว


---
## Section 4 — Path B: HuggingFace Transformers (DETR)

ใน section นี้:
1. **4a.** ใช้ pretrained DETR (`facebook/detr-resnet-50`) inference
2. **4b.** Demo pretrained RT-DETR (model ใหม่กว่า ไม่ใช้ NMS)
3. **4c.** Fine-tune DETR บน pothole dataset
4. **4d.** ประเมินผลด้วย `torchmetrics`

**ทำไมเลือก DETR ไม่ใช่ RT-DETR สำหรับ fine-tuning?** RT-DETR ใช้ memory เยอะกว่า — บน Colab T4 free มักจะ OOM ระหว่าง fine-tune ส่วน DETR fine-tune ได้สบายด้วย batch=4 + gradient accumulation


### 4a. Pretrained DETR inference

DETR pretrained บน COCO เช่นเดียวกับ YOLO — ก็จะไม่ตรวจเจอ pothole เช่นกัน


In [ ]:
# @title Load DETR pretrained + inference
from transformers import AutoImageProcessor, AutoModelForObjectDetection

DETR_CKPT = "facebook/detr-resnet-50"
processor_detr_pre = AutoImageProcessor.from_pretrained(DETR_CKPT)
model_detr_pre = AutoModelForObjectDetection.from_pretrained(DETR_CKPT).cuda().eval()

DETR_COCO_NAMES = model_detr_pre.config.id2label  # {0: 'N/A', 1: 'person', ...}

@torch.no_grad()
def detr_predict(model, processor, image_path, threshold=0.5):
    """คืน boxes (xyxy), scores, labels จาก DETR-style model"""
    img = PILImage.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(model.device)
    outputs = model(**inputs)
    target_sizes = torch.tensor([img.size[::-1]])  # (H, W)
    results = processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=target_sizes
    )[0]
    return results, img

# Inference + visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img_path in zip(axes, TEST_IMAGES):
    results, img = detr_predict(model_detr_pre, processor_detr_pre, img_path, threshold=0.7)
    img_arr = np.array(img)

    if len(results["boxes"]):
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=results["labels"].cpu().numpy(),
        )
        labels = [f"{DETR_COCO_NAMES[int(c)]} {s:.2f}"
                  for c, s in zip(results["labels"], results["scores"])]
        annotated = box_annotator.annotate(img_arr.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img_arr

    ax.imshow(annotated)
    ax.set_title(f"Pretrained DETR\n{Path(img_path).name}")
    ax.axis("off")

plt.tight_layout(); plt.show()
print("\n📌 DETR pretrained ก็ตรวจ pothole ไม่เจอเช่นกัน เพราะ COCO ไม่มี class นี้")


### 4b. Pretrained RT-DETR inference (demo only)

**RT-DETR** เป็น real-time DETR variant ที่
- ลบ NMS ออกได้
- เร็วกว่า DETR ปกติ
- มี backbone หลายขนาด (R18 / R34 / R50 / R101)

เราใช้ **R18 backbone** เพราะเล็กพอที่จะ inference บน T4 ได้ แต่จะไม่ fine-tune (ใหญ่เกินไปสำหรับ T4 free)


In [ ]:
# @title Pretrained RT-DETR demo
from transformers import RTDetrImageProcessor, RTDetrForObjectDetection

RTDETR_CKPT = "PekingU/rtdetr_r18vd_coco_o365"
processor_rtdetr = RTDetrImageProcessor.from_pretrained(RTDETR_CKPT)
model_rtdetr = RTDetrForObjectDetection.from_pretrained(RTDETR_CKPT).cuda().eval()

RTDETR_NAMES = model_rtdetr.config.id2label

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img_path in zip(axes, TEST_IMAGES):
    results, img = detr_predict(model_rtdetr, processor_rtdetr, img_path, threshold=0.5)
    img_arr = np.array(img)

    if len(results["boxes"]):
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=results["labels"].cpu().numpy(),
        )
        labels = [f"{RTDETR_NAMES.get(int(c), str(int(c)))} {s:.2f}"
                  for c, s in zip(results["labels"], results["scores"])]
        annotated = box_annotator.annotate(img_arr.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img_arr

    ax.imshow(annotated)
    ax.set_title(f"Pretrained RT-DETR\n{Path(img_path).name}")
    ax.axis("off")

plt.tight_layout(); plt.show()

# Free memory
del model_rtdetr
torch.cuda.empty_cache()


### 4c. Fine-tune DETR บน pothole dataset

**ขั้นตอน:**
1. Load DETR fresh พร้อม `num_labels=1` และ freeze backbone (2-stage training)
2. สร้าง Albumentations pipeline พร้อม augmentation สำหรับภาพถนน
3. เขียน collate_fn ที่ pad ภาพให้ขนาดเท่ากันใน batch
4. **Stage 1:** ฝึก head อย่างเดียว (backbone frozen) ด้วย LR สูง
5. **Stage 2:** Unfreeze backbone แล้ว fine-tune ทั้งโมเดลด้วย LR ต่ำ
6. ใช้ fp16, cosine LR schedule, early stopping, และ save best checkpoint

**ทำไม 2-stage?** ช่วยให้ head converge ก่อน — ถ้า unfreeze backbone ตั้งแต่แรกด้วย random head จะทำให้ gradient ไม่เสถียรและ backbone ถูกทำลาย

In [ ]:
# @title โหลด DETR fresh สำหรับ fine-tuning
ID2LABEL = {0: "pothole"}
LABEL2ID = {"pothole": 0}

# ใช้ processor แยกตัว (อาจปรับเปลี่ยน)
processor_detr = AutoImageProcessor.from_pretrained(
    DETR_CKPT,
    do_resize=True,
    size={"shortest_edge": 600, "longest_edge": 1000},
)

# โหลด model พร้อม head ใหม่
model_detr = AutoModelForObjectDetection.from_pretrained(
    DETR_CKPT,
    num_labels=1,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# Freeze backbone — ฝึก head ก่อนแล้วค่อย unfreeze (2-stage training)
for name, param in model_detr.named_parameters():
    if "backbone" in name:
        param.requires_grad = False

n_total = sum(p.numel() for p in model_detr.parameters()) / 1e6
n_train = sum(p.numel() for p in model_detr.parameters() if p.requires_grad) / 1e6
print(f"✓ โหลด DETR — {n_total:.1f}M params total, {n_train:.1f}M trainable (backbone frozen)")
print(f"✓ Classes: {ID2LABEL}")

In [ ]:
# @title Albumentations transform
import albumentations as A

# Train transform — augmentation หลากหลายขึ้นสำหรับ dataset เล็ก
train_aug = A.Compose([
    A.Resize(width=600, height=600),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.3),
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.2),
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=0.2),  # เงาบนถนน
    A.CLAHE(clip_limit=2.0, p=0.2),  # ช่วยเรื่อง contrast ในภาพถนน
], bbox_params=A.BboxParams(format="coco", label_fields=["category"], min_area=1.0))

val_aug = A.Compose([
    A.Resize(width=600, height=600),
], bbox_params=A.BboxParams(format="coco", label_fields=["category"], min_area=1.0))

print("✓ Albumentations pipelines พร้อม (format=coco, เพิ่ม augmentation สำหรับถนน)")

In [ ]:
# @title Format annotations + transform pipeline
def format_annotations_coco_style(image_id, categories, areas, bboxes):
    """แปลง list ของ bbox ให้เป็น COCO annotation dicts สำหรับ DETR processor"""
    return [
        {"image_id": image_id, "category_id": cat, "iscrowd": 0,
         "area": ar, "bbox": list(b)}
        for cat, ar, b in zip(categories, areas, bboxes)
    ]


def make_transform(aug_pipeline, image_processor):
    """Factory: คืนฟังก์ชัน transform ที่ใช้กับ HF dataset"""
    def transform_batch(examples):
        images, targets = [], []
        for img, image_id, objs in zip(
            examples["image"], examples["image_id"], examples["objects"]
        ):
            img_np = np.array(img.convert("RGB"))
            out = aug_pipeline(
                image=img_np,
                bboxes=objs["bbox"],
                category=objs["category"],
            )
            images.append(out["image"])
            anns = format_annotations_coco_style(
                image_id=image_id,
                categories=out["category"],
                # คำนวณ area ใหม่หลัง resize
                areas=[b[2] * b[3] for b in out["bboxes"]],
                bboxes=out["bboxes"],
            )
            targets.append({"image_id": image_id, "annotations": anns})

        encoded = image_processor(images=images, annotations=targets, return_tensors="pt")
        return encoded
    return transform_batch


train_ds = hf_dataset["train"].with_transform(make_transform(train_aug, processor_detr))
val_ds   = hf_dataset["validation"].with_transform(make_transform(val_aug, processor_detr))

# ตรวจสอบ shape
sample = train_ds[0]
print(f"✓ pixel_values shape: {sample['pixel_values'].shape}")
print(f"✓ labels keys: {list(sample['labels'].keys())}")
print(f"✓ จำนวน boxes ในตัวอย่างแรก: {len(sample['labels']['boxes'])}")


In [ ]:
def collate_fn(batch):
    import torch

    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    pixel_mask   = torch.stack([item["pixel_mask"] for item in batch])

    # Keep only required fields for DETR
    labels = []
    for item in batch:
        l = item["labels"]
        labels.append({
            "class_labels": l["class_labels"],
            "boxes": l["boxes"],
        })

    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "labels": labels,
    }

In [ ]:
# @title TrainingArguments + Trainer + ฝึก (2-stage: frozen backbone → full fine-tune)
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

OUT_DETR = RUNS_DIR / "detr_pothole"

# ---------- Stage 1: Backbone frozen — ฝึก head 15 epochs ----------
args_stage1 = TrainingArguments(
    output_dir=str(OUT_DETR / "stage1"),
    num_train_epochs=15,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,   # effective batch = 8
    learning_rate=5e-4,              # LR สูงขึ้นได้เพราะ backbone frozen
    weight_decay=1e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=0.1,               # DETR ต้องการ gradient clipping ต่ำ
    dataloader_num_workers=2,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    fp16=True,
    logging_nan_inf_filter=True,
)

# ---------- Stage 2: Unfreeze backbone — full fine-tune 20 epochs ----------
args_stage2 = TrainingArguments(
    output_dir=str(OUT_DETR / "stage2"),
    num_train_epochs=20,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,   # effective batch = 8
    learning_rate=1e-5,              # LR ต่ำลงสำหรับ backbone
    weight_decay=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_grad_norm=0.1,
    dataloader_num_workers=2,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    fp16=True,
    logging_nan_inf_filter=True,
)

if FAST_MODE:
    print("⚠️ FAST_MODE: ข้าม fine-tuning DETR")
    detr_train_time = 0
else:
    t0 = time.time()

    # --- Stage 1: frozen backbone ---
    print("🔷 Stage 1: ฝึก head (backbone frozen)")
    print(f"  epochs={args_stage1.num_train_epochs}, lr={args_stage1.learning_rate}, "
          f"batch={args_stage1.per_device_train_batch_size}×{args_stage1.gradient_accumulation_steps}")

    trainer_s1 = Trainer(
        model=model_detr,
        args=args_stage1,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )
    trainer_s1.train()
    stage1_time = time.time() - t0
    print(f"✓ Stage 1 เสร็จใน {stage1_time/60:.1f} นาที")

    # --- Stage 2: unfreeze backbone ---
    print("\n🔶 Stage 2: Unfreeze backbone — full fine-tune")
    for param in model_detr.parameters():
        param.requires_grad = True

    n_train = sum(p.numel() for p in model_detr.parameters() if p.requires_grad) / 1e6
    print(f"  trainable params: {n_train:.1f}M")
    print(f"  epochs={args_stage2.num_train_epochs}, lr={args_stage2.learning_rate}")

    trainer_s2 = Trainer(
        model=model_detr,
        args=args_stage2,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )
    trainer_s2.train()
    detr_train_time = time.time() - t0
    print(f"\n✓ ฝึก DETR ทั้ง 2 stages เสร็จใน {detr_train_time/60:.1f} นาที")

model_detr.eval()

### 4d. ประเมิน DETR ด้วย torchmetrics

In [ ]:
# @title ประเมิน mAP บน validation set
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def evaluate_detr(model, processor, hf_val_ds, threshold=0.0):
    """ประเมิน DETR-style model ด้วย torchmetrics MeanAveragePrecision"""
    model.eval()
    metric = MeanAveragePrecision(box_format="xyxy")

    for example in hf_val_ds:
        img = example["image"].convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(model.device)
        outputs = model(**inputs)
        target_sizes = torch.tensor([img.size[::-1]])  # (H, W)
        results = processor.post_process_object_detection(
            outputs, threshold=threshold, target_sizes=target_sizes
        )[0]

        # Predictions
        preds = [{
            "boxes":  results["boxes"].cpu(),
            "scores": results["scores"].cpu(),
            "labels": results["labels"].cpu(),
        }]

        # Ground truth (COCO bbox xywh → xyxy)
        gt_xyxy = []
        for b in example["objects"]["bbox"]:
            gt_xyxy.append([b[0], b[1], b[0]+b[2], b[1]+b[3]])
        targets = [{
            "boxes":  torch.tensor(gt_xyxy, dtype=torch.float32),
            "labels": torch.tensor(example["objects"]["category"], dtype=torch.int64),
        }]
        metric.update(preds, targets)

    return metric.compute()


# Evaluate fine-tuned DETR (ถ้า FAST_MODE จะใช้ pretrained-with-new-head)
print("กำลังประเมิน fine-tuned DETR...")
model_detr.cuda()
detr_metrics_raw = evaluate_detr(model_detr, processor_detr, hf_dataset["validation"])

detr_metrics = {
    "map50": float(detr_metrics_raw["map_50"]),
    "map":   float(detr_metrics_raw["map"]),
}
print(f"📊 Fine-tuned DETR:")
print(f"  mAP@0.5      : {detr_metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95 : {detr_metrics['map']:.4f}")


In [ ]:
# @title Visualize fine-tuned DETR predictions บน TEST_IMAGES
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img_path in zip(axes, TEST_IMAGES):
    results, img = detr_predict(model_detr, processor_detr, img_path, threshold=0.5)
    img_arr = np.array(img)

    if len(results["boxes"]):
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=results["labels"].cpu().numpy(),
        )
        labels = [f"pothole {s:.2f}" for s in results["scores"]]
        annotated = box_annotator.annotate(img_arr.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img_arr

    ax.imshow(annotated)
    ax.set_title(f"Fine-tuned DETR\n{Path(img_path).name}")
    ax.axis("off")

plt.tight_layout(); plt.show()


**สิ่งที่ได้เรียนรู้**
- **2-stage training** (frozen backbone → full fine-tune) ช่วยให้ head converge ก่อนจึงค่อย update backbone — ป้องกัน catastrophic forgetting
- **fp16 + gradient clipping ต่ำ (0.1)** จำเป็นสำหรับ DETR เพราะ loss ของ Hungarian matching มี gradient ที่ไม่เสถียร
- **Augmentation หลากหลาย** (shadow, noise, CLAHE) สำคัญมากกับ dataset เล็ก — ช่วยลด overfitting
- **Eval ระหว่าง train + early stopping** ช่วยจับจุดที่ model เริ่ม overfit และ save best checkpoint อัตโนมัติ
- ใช้ `torchmetrics` ประเมิน mAP เพื่อให้เปรียบเทียบกับ YOLO ได้แบบ apples-to-apples

---
## Section 5 — เปรียบเทียบ / Head-to-head

รวบรวมตัวเลขทั้งหมดมาเทียบกัน — ทั้ง mAP, จำนวน parameters, FPS และเวลาฝึก


In [ ]:
# @title คำนวณ FPS และจำนวน parameters
def measure_fps(predict_fn, n_runs=20, warmup=3):
    """วัด FPS เฉลี่ยจากการรัน predict_fn n ครั้ง (warmup ครั้งแรกไม่นับ)"""
    test_img = TEST_IMAGES[0]
    # warmup
    for _ in range(warmup):
        predict_fn(test_img)
    torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(n_runs):
        predict_fn(test_img)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    return n_runs / elapsed

# YOLO predict
def yolo_predict_fn(img_path):
    return model_yolo_ft(img_path, verbose=False, conf=0.25)

# DETR predict
def detr_predict_fn(img_path):
    img = PILImage.open(img_path).convert("RGB")
    inputs = processor_detr(images=img, return_tensors="pt").to(model_detr.device)
    with torch.no_grad():
        return model_detr(**inputs)

print("กำลังวัด FPS...")
yolo_fps = measure_fps(yolo_predict_fn)
detr_fps = measure_fps(detr_predict_fn)

# Parameters
yolo_params = sum(p.numel() for p in model_yolo_ft.model.parameters()) / 1e6
detr_params = sum(p.numel() for p in model_detr.parameters()) / 1e6

print(f"\n📊 FPS และ size:")
print(f"  YOLO11n: {yolo_fps:.1f} FPS, {yolo_params:.1f}M params")
print(f"  DETR:    {detr_fps:.1f} FPS, {detr_params:.1f}M params")


In [ ]:
# @title สรุปตารางเปรียบเทียบ
import pandas as pd

# Pretrained models ตรวจ pothole ไม่เจอเลย → mAP = 0
comparison_df = pd.DataFrame([
    {
        "Model": "YOLO11n (pretrained, COCO)",
        "mAP@0.5": 0.0,
        "mAP@0.5:0.95": 0.0,
        "Params (M)": yolo_params,
        "FPS (T4)": yolo_fps,
        "Train time": "—",
    },
    {
        "Model": "YOLO11n (fine-tuned)",
        "mAP@0.5": yolo_metrics["map50"],
        "mAP@0.5:0.95": yolo_metrics["map"],
        "Params (M)": yolo_params,
        "FPS (T4)": yolo_fps,
        "Train time": f"{yolo_train_time/60:.1f} min" if not FAST_MODE else "—",
    },
    {
        "Model": "DETR (pretrained, COCO)",
        "mAP@0.5": 0.0,
        "mAP@0.5:0.95": 0.0,
        "Params (M)": detr_params,
        "FPS (T4)": detr_fps,
        "Train time": "—",
    },
    {
        "Model": "DETR (fine-tuned)",
        "mAP@0.5": detr_metrics["map50"],
        "mAP@0.5:0.95": detr_metrics["map"],
        "Params (M)": detr_params,
        "FPS (T4)": detr_fps,
        "Train time": "-" #f"{detr_train_time/60:.1f} min" if not FAST_MODE else "—",
    },
])

# Format
def fmt_row(row):
    return {
        "Model": row["Model"],
        "mAP@0.5": f"{row['mAP@0.5']:.3f}",
        "mAP@0.5:0.95": f"{row['mAP@0.5:0.95']:.3f}",
        "Params (M)": f"{row['Params (M)']:.1f}",
        "FPS (T4)": f"{row['FPS (T4)']:.1f}",
        "Train time": row["Train time"],
    }

display_df = pd.DataFrame([fmt_row(r) for _, r in comparison_df.iterrows()])
print(display_df.to_string(index=False))


In [ ]:
# @title Visualize 4-column comparison บน TEST_IMAGES
fig, axes = plt.subplots(3, 4, figsize=(20, 12))

col_titles = [
    "YOLO11n\nPretrained",
    "YOLO11n\nFine-tuned",
    "DETR\nPretrained",
    "DETR\nFine-tuned",
]

for col_idx, ax in enumerate(axes[0]):
    ax.set_title(col_titles[col_idx], fontsize=14, fontweight="bold")

for row_idx, img_path in enumerate(TEST_IMAGES):
    img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    # Column 0: Pretrained YOLO
    res = model_yolo_pre(img_path, verbose=False, conf=0.25)
    annotated = img_rgb.copy()
    boxes_xyxy, confs, cls_ids = [], [], []
    for r in res:
        for b in r.boxes:
            boxes_xyxy.append(b.xyxy[0].cpu().numpy())
            confs.append(float(b.conf[0]))
            cls_ids.append(int(b.cls[0]))
    if boxes_xyxy:
        det = sv.Detections(xyxy=np.array(boxes_xyxy),
                            confidence=np.array(confs),
                            class_id=np.array(cls_ids))
        labels = [f"{COCO_NAMES[c]} {s:.2f}" for c, s in zip(cls_ids, confs)]
        annotated = box_annotator.annotate(annotated, det)
        annotated = label_annotator.annotate(annotated, det, labels)
    axes[row_idx, 0].imshow(annotated); axes[row_idx, 0].axis("off")

    # Column 1: Fine-tuned YOLO
    res = model_yolo_ft(img_path, verbose=False, conf=0.25)
    annotated = img_rgb.copy()
    boxes_xyxy, confs, cls_ids = [], [], []
    for r in res:
        for b in r.boxes:
            boxes_xyxy.append(b.xyxy[0].cpu().numpy())
            confs.append(float(b.conf[0]))
            cls_ids.append(int(b.cls[0]))
    if boxes_xyxy:
        det = sv.Detections(xyxy=np.array(boxes_xyxy),
                            confidence=np.array(confs),
                            class_id=np.array(cls_ids))
        labels = [f"pothole {s:.2f}" for s in confs]
        annotated = box_annotator.annotate(annotated, det)
        annotated = label_annotator.annotate(annotated, det, labels)
    axes[row_idx, 1].imshow(annotated); axes[row_idx, 1].axis("off")

    # Column 2: Pretrained DETR
    results, _ = detr_predict(model_detr_pre, processor_detr_pre, img_path, threshold=0.7)
    annotated = img_rgb.copy()
    if len(results["boxes"]):
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=results["labels"].cpu().numpy(),
        )
        labels = [f"{DETR_COCO_NAMES[int(c)]} {s:.2f}"
                  for c, s in zip(results["labels"], results["scores"])]
        annotated = box_annotator.annotate(annotated, det)
        annotated = label_annotator.annotate(annotated, det, labels)
    axes[row_idx, 2].imshow(annotated); axes[row_idx, 2].axis("off")

    # Column 3: Fine-tuned DETR
    results, _ = detr_predict(model_detr, processor_detr, img_path, threshold=0.5)
    annotated = img_rgb.copy()
    if len(results["boxes"]):
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=results["labels"].cpu().numpy(),
        )
        labels = [f"pothole {s:.2f}" for s in results["scores"]]
        annotated = box_annotator.annotate(annotated, det)
        annotated = label_annotator.annotate(annotated, det, labels)
    axes[row_idx, 3].imshow(annotated); axes[row_idx, 3].axis("off")

    axes[row_idx, 0].set_ylabel(Path(img_path).name, fontsize=10)

plt.tight_layout(); plt.show()


**การตีความผลลัพธ์**

- **Pretrained ทั้งสอง model ตรวจ pothole ไม่เจอ** — เพราะ COCO ไม่มี class นี้ → fine-tune จึงจำเป็น
- **YOLO ฝึกเร็วกว่า** (~5–7 นาที) แต่ DETR ก็ใช้ได้ใน T4 free ถ้า batch เล็กพอ
- **YOLO เร็วกว่า DETR ที่ inference** (FPS สูงกว่า) — เหมาะกับ deployment แบบ real-time
- **DETR ไม่ใช้ NMS** — สถาปัตยกรรมเรียบง่ายกว่าตอน deployment
- **ทั้งสองได้ mAP ใกล้เคียงกัน** บน dataset ขนาดเล็กแบบนี้ — แต่ DETR มักจะดีกว่าบนวัตถุที่ทับซ้อนกันใน dataset ใหญ่กว่า


---
## Section 6 — Zero-shot detection (จุดแข็งของ transformer)

ลองใช้ transformer-based detectors ที่**ไม่ต้องฝึก** — แค่ใช้ text prompt ก็ตรวจวัตถุได้

ใน section นี้เราจะ:
1. ใช้ **OWLv2** (`google/owlv2-base-patch16-ensemble`) — รับ text prompt เป็น list ของ class names
2. ใช้ **Grounding DINO** (`IDEA-Research/grounding-dino-tiny`) — รับ phrase prompt ที่เป็นประโยคได้
3. ทำ **prompt engineering** — ลอง prompt หลายแบบดูว่าอันไหนได้ผลดีที่สุด
4. คำนวณ zero-shot mAP เทียบกับ fine-tuned models


### 6a. OWLv2 + prompt sweep

In [ ]:
# @title Load OWLv2
from transformers import Owlv2Processor, Owlv2ForObjectDetection

OWLV2_CKPT = "google/owlv2-base-patch16-ensemble"
processor_owl = Owlv2Processor.from_pretrained(OWLV2_CKPT)
model_owl = Owlv2ForObjectDetection.from_pretrained(OWLV2_CKPT).cuda().eval()

print(f"✓ Loaded OWLv2 ({sum(p.numel() for p in model_owl.parameters())/1e6:.1f}M params)")


In [ ]:
# @title OWLv2 inference function
@torch.no_grad()
def owlv2_predict(image_path, prompts, threshold=0.1):
    """
    image_path: path string
    prompts: list of strings เช่น ['pothole', 'hole in road']
    คืน list ของ {prompt, boxes, scores} per prompt
    """
    img = PILImage.open(image_path).convert("RGB")
    inputs = processor_owl(images=img, text=[prompts], return_tensors="pt").to(model_owl.device)
    outputs = model_owl(**inputs)
    target_sizes = torch.tensor([img.size[::-1]])
    results = processor_owl.post_process_grounded_object_detection(
        outputs=outputs, threshold=threshold, target_sizes=target_sizes
    )[0]
    return results, img


In [ ]:
# @title Prompt sweep บน TEST_IMAGES[0]
PROMPTS_TO_TRY = [
    "pothole",
    "hole in the road",
    "damaged road surface",
    "crack on asphalt",
    "spaceship",   # negative — should find nothing
]

img_path = TEST_IMAGES[0]
fig, axes = plt.subplots(1, len(PROMPTS_TO_TRY), figsize=(5*len(PROMPTS_TO_TRY), 5))

for ax, prompt in zip(axes, PROMPTS_TO_TRY):
    results, img = owlv2_predict(img_path, [prompt], threshold=0.1)
    img_arr = np.array(img)

    n_det = len(results["boxes"])
    if n_det:
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=np.zeros(n_det, dtype=int),
        )
        labels = [f"{s:.2f}" for s in results["scores"]]
        annotated = box_annotator.annotate(img_arr.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img_arr

    ax.imshow(annotated)
    ax.set_title(f'"{prompt}"\n→ {n_det} detections', fontsize=11)
    ax.axis("off")

plt.suptitle("OWLv2 prompt sweep (no training)", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

print("\n📌 สังเกต: prompt ที่ specific มากเกินไปอาจจะ recall ต่ำ")
print("   prompt ที่กว้างไปก็อาจจะ false positive เยอะ")
print("   prompt 'spaceship' ไม่เจออะไรเลย — ตามที่คาด")


### 6b. Grounding DINO + phrase prompts

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

GDINO_CKPT = "IDEA-Research/grounding-dino-tiny"
processor_gdino = AutoProcessor.from_pretrained(GDINO_CKPT)
model_gdino = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_CKPT).cuda().eval()

@torch.no_grad()
def gdino_predict(image_path, text, box_threshold=0.25, text_threshold=0.25):
    """text: ประโยคเดียว ลงท้ายด้วย period เช่น 'a pothole.'"""
    img = PILImage.open(image_path).convert("RGB")
    inputs = processor_gdino(images=img, text=text, return_tensors="pt").to(model_gdino.device)
    outputs = model_gdino(**inputs)
    target_sizes = [img.size[::-1]]
    results = processor_gdino.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=target_sizes,
    )[0]
    return results, img


PHRASE_PROMPTS = [
    "a pothole.",
    "a hole in the road.",
    "damage on the road surface.",
    "a parked car on the side.",   # context-specific phrase
]

img_path = TEST_IMAGES[1]
fig, axes = plt.subplots(1, len(PHRASE_PROMPTS), figsize=(5*len(PHRASE_PROMPTS), 5))

for ax, phrase in zip(axes, PHRASE_PROMPTS):
    results, img = gdino_predict(img_path, phrase)
    img_arr = np.array(img)

    n_det = len(results["boxes"])
    if n_det:
        det = sv.Detections(
            xyxy=results["boxes"].cpu().numpy(),
            confidence=results["scores"].cpu().numpy(),
            class_id=np.zeros(n_det, dtype=int),
        )
        labels = [f"{s:.2f}" for s in results["scores"]]
        annotated = box_annotator.annotate(img_arr.copy(), det)
        annotated = label_annotator.annotate(annotated, det, labels)
    else:
        annotated = img_arr

    ax.imshow(annotated)
    ax.set_title(f'"{phrase}"\n→ {n_det} detections', fontsize=11)
    ax.axis("off")

plt.suptitle("Grounding DINO phrase prompts (no training)", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### 6c. Zero-shot mAP (เทียบกับ fine-tuned models)

วัด mAP ของ OWLv2 บน validation set โดยใช้ prompt `"pothole"` — เพื่อดูว่า zero-shot สู้ fine-tuned ได้แค่ไหน


In [ ]:
# @title คำนวณ OWLv2 zero-shot mAP บน validation set
@torch.no_grad()
def evaluate_owlv2(model, processor, hf_val_ds, prompt="pothole", threshold=0.1):
    metric = MeanAveragePrecision(box_format="xyxy")

    for example in hf_val_ds:
        img = example["image"].convert("RGB")
        inputs = processor(images=img, text=[[prompt]], return_tensors="pt").to(model.device)
        outputs = model(**inputs)
        target_sizes = torch.tensor([img.size[::-1]])
        results = processor.post_process_grounded_object_detection(
            outputs=outputs, threshold=threshold, target_sizes=target_sizes
        )[0]

        preds = [{
            "boxes":  results["boxes"].cpu(),
            "scores": results["scores"].cpu(),
            # OWLv2 คืน label index ของ prompt — เรา map ทั้งหมดเป็น class 0
            "labels": torch.zeros(len(results["boxes"]), dtype=torch.int64),
        }]

        gt_xyxy = [[b[0], b[1], b[0]+b[2], b[1]+b[3]] for b in example["objects"]["bbox"]]
        targets = [{
            "boxes":  torch.tensor(gt_xyxy, dtype=torch.float32),
            "labels": torch.tensor(example["objects"]["category"], dtype=torch.int64),
        }]
        metric.update(preds, targets)

    return metric.compute()


print("กำลังประเมิน OWLv2 zero-shot บน val set (อาจใช้เวลาสักครู่)...")
owl_metrics_raw = evaluate_owlv2(model_owl, processor_owl, hf_dataset["validation"],
                                   prompt="pothole", threshold=0.1)
owl_metrics = {
    "map50": float(owl_metrics_raw["map_50"]),
    "map":   float(owl_metrics_raw["map"]),
}

print(f"\n📊 OWLv2 zero-shot (prompt='pothole'):")
print(f"  mAP@0.5      : {owl_metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95 : {owl_metrics['map']:.4f}")

# เพิ่มเข้าตารางเปรียบเทียบ
extra_row = pd.DataFrame([{
    "Model": "OWLv2 (zero-shot, prompt='pothole')",
    "mAP@0.5": f"{owl_metrics['map50']:.3f}",
    "mAP@0.5:0.95": f"{owl_metrics['map']:.3f}",
    "Params (M)": f"{sum(p.numel() for p in model_owl.parameters())/1e6:.1f}",
    "FPS (T4)": "—",
    "Train time": "0 (no training)",
}])
final_df = pd.concat([display_df, extra_row], ignore_index=True)
print("\n📊 ตารางเปรียบเทียบสุดท้าย:")
print(final_df.to_string(index=False))


**สิ่งที่ได้เรียนรู้**
- **Zero-shot detectors ตรวจ pothole ได้โดยไม่ต้องฝึก** ผ่าน text prompt — น่าทึ่งมากเพราะไม่เคยเห็น pothole ใน training data
- **Prompt engineering สำคัญ** — `"pothole"` กับ `"hole in the road"` ให้ผลต่างกัน
- **Trade-off ที่ชัดเจน:** zero-shot แลก mAP กับการไม่ต้อง label data
- **เมื่อไหร่ใช้ zero-shot ดีกว่า:** ตอนต้นโครงการ (ก่อน label), หรือเมื่อต้องตรวจ class ใหม่ที่ไม่มี data


---
## Section 7 — Gemini API สำหรับ Pothole Detection

ในส่วนนี้เราจะใช้ **Google Gemini API** (multimodal LLM) เพื่อตรวจจับ pothole จากภาพ โดยไม่ต้องฝึกโมเดลเลย — ส่งภาพพร้อม prompt แล้วให้โมเดลตอบกลับมาเป็น bounding box + confidence

### ข้อดี
- ไม่ต้อง fine-tune, ไม่ต้องมี GPU สำหรับ inference
- รองรับ reasoning เกี่ยวกับภาพแบบ open-ended
- ใช้ได้กับ class ใหม่ ๆ โดยแค่เปลี่ยน prompt

### ข้อจำกัด
- ต้องมี API key และมีค่าใช้จ่ายต่อ request
- ความเร็วช้ากว่า on-device model (YOLO, DETR)
- ผลลัพธ์อาจไม่ consistent เท่า fine-tuned model

In [ ]:
# @title ติดตั้ง Google Generative AI SDK
!pip install -q "google-genai>=1.0"

In [ ]:
# @title ตั้งค่า Gemini API key
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")  # ตั้งค่าใน Colab Secrets
client = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODEL = "gemini-2.5-flash"  # ใช้ flash สำหรับความเร็วและต้นทุนต่ำ
print(f"✓ Gemini client ready — model: {GEMINI_MODEL}")

In [ ]:
# @title ฟังก์ชัน Gemini pothole detection
import json
from PIL import Image as PILImage

GEMINI_PROMPT = """Detect all potholes in this road image.
Return a JSON array where each element has:
  - "box": [xmin, ymin, xmax, ymax] in pixel coordinates
  - "confidence": a float between 0 and 1
  - "description": a short description of the pothole

If no potholes are found, return an empty array [].
Return ONLY the JSON array, no other text."""


def gemini_detect(image_path, prompt=GEMINI_PROMPT, model=GEMINI_MODEL):
    """
    ส่งภาพไปยัง Gemini API แล้ว parse ผลลัพธ์เป็น list of detections
    Returns: list of dict with keys 'box', 'confidence', 'description'
    """
    img = PILImage.open(image_path).convert("RGB")

    response = client.models.generate_content(
        model=model,
        contents=[prompt, img],
    )

    # Parse JSON จาก response
    text = response.text.strip()
    # ลบ markdown code fence ถ้ามี
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]

    try:
        detections = json.loads(text)
    except json.JSONDecodeError:
        print(f"⚠️ ไม่สามารถ parse JSON ได้: {text[:200]}")
        detections = []

    return detections


# ทดสอบกับภาพแรก
test_result = gemini_detect(TEST_IMAGES[0])
print(f"พบ {len(test_result)} pothole(s) ในภาพทดสอบ")
for i, det in enumerate(test_result):
    print(f"  #{i+1}: confidence={det['confidence']:.2f}, box={det['box']}, {det.get('description', '')}")

In [ ]:
# @title Visualize Gemini detections บน TEST_IMAGES
import supervision as sv
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Gemini API Pothole Detection", fontsize=14, fontweight="bold")

for idx, img_path in enumerate(TEST_IMAGES):
    img = PILImage.open(img_path).convert("RGB")
    img_np = np.array(img)

    detections = gemini_detect(img_path)

    if detections:
        boxes = np.array([d["box"] for d in detections])
        confidences = np.array([d["confidence"] for d in detections])
        sv_detections = sv.Detections(
            xyxy=boxes,
            confidence=confidences,
            class_id=np.zeros(len(boxes), dtype=int),
        )
        annotator = sv.BoxAnnotator(thickness=2)
        label_annotator = sv.LabelAnnotator(text_scale=0.5)
        labels = [f"pothole {d['confidence']:.2f}" for d in detections]
        img_np = annotator.annotate(img_np.copy(), sv_detections)
        img_np = label_annotator.annotate(img_np, sv_detections, labels=labels)

    axes[idx].imshow(img_np)
    axes[idx].set_title(f"Gemini — {len(detections)} detection(s)")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# @title คำนวณ Gemini mAP บน validation set (sample)
from torchmetrics.detection import MeanAveragePrecision
import time

MAX_EVAL_SAMPLES = 50  # จำกัดจำนวนเพื่อควบคุมค่า API

metric = MeanAveragePrecision(iou_type="bbox")
val_ds = hf_dataset["validation"]
n_samples = min(MAX_EVAL_SAMPLES, len(val_ds))

print(f"กำลังประเมิน Gemini บน {n_samples} ภาพ (จำกัดเพื่อควบคุมค่า API)...")
start_time = time.time()

for i in range(n_samples):
    sample = val_ds[i]
    img = sample["image"].convert("RGB")
    w, h = img.size

    # Ground truth
    gt_boxes = torch.tensor(sample["objects"]["bbox"], dtype=torch.float32)
    # COCO (x,y,w,h) → xyxy
    if len(gt_boxes) > 0:
        gt_xyxy = gt_boxes.clone()
        gt_xyxy[:, 2] = gt_boxes[:, 0] + gt_boxes[:, 2]
        gt_xyxy[:, 3] = gt_boxes[:, 1] + gt_boxes[:, 3]
    else:
        gt_xyxy = torch.zeros((0, 4))

    gt_labels = torch.zeros(len(gt_xyxy), dtype=torch.long)

    # Gemini predictions
    try:
        dets = gemini_detect(img_path)
    except Exception as e:
        print(f"  ⚠️ ภาพ {i}: error — {e}")
        dets = []

    if dets:
        pred_boxes = torch.tensor([d["box"] for d in dets], dtype=torch.float32)
        # Clamp to image bounds
        pred_boxes[:, [0, 2]] = pred_boxes[:, [0, 2]].clamp(0, w)
        pred_boxes[:, [1, 3]] = pred_boxes[:, [1, 3]].clamp(0, h)
        pred_scores = torch.tensor([d["confidence"] for d in dets], dtype=torch.float32)
        pred_labels = torch.zeros(len(dets), dtype=torch.long)
    else:
        pred_boxes = torch.zeros((0, 4))
        pred_scores = torch.zeros(0)
        pred_labels = torch.zeros(0, dtype=torch.long)

    metric.update(
        [dict(boxes=pred_boxes, scores=pred_scores, labels=pred_labels)],
        [dict(boxes=gt_xyxy, labels=gt_labels)],
    )

    if (i + 1) % 10 == 0:
        print(f"  ประเมินแล้ว {i+1}/{n_samples} ภาพ")

elapsed = time.time() - start_time
gemini_metrics = metric.compute()

print(f"\n📊 Gemini ({GEMINI_MODEL}) บน {n_samples} ภาพ:")
print(f"  mAP@0.5      : {gemini_metrics['map_50']:.4f}")
print(f"  mAP@0.5:0.95 : {gemini_metrics['map']:.4f}")
print(f"  เวลาทั้งหมด  : {elapsed:.1f}s ({elapsed/n_samples:.2f}s/ภาพ)")

**สิ่งที่ได้เรียนรู้**
- **Gemini API สามารถตรวจจับ pothole ได้** ผ่าน multimodal prompting โดยไม่ต้องฝึกโมเดล
- **Prompt engineering สำคัญมาก** — การระบุ output format ชัดเจน (JSON) ช่วยให้ parse ผลลัพธ์ได้ง่าย
- **ข้อจำกัดหลัก:** ความเร็วและค่าใช้จ่าย API ทำให้ไม่เหมาะกับ real-time หรือ large-scale inference
- **เหมาะกับ:** rapid prototyping, การตรวจสอบภาพทีละรูป, หรือเป็น teacher model สำหรับ knowledge distillation

---
## Section 8 — สรุป / Wrap-up

### เลือกใช้เส้นทางไหนดี?

- **YOLO11** — เหมาะที่สุดเมื่อต้องการ deployment เร็ว, edge device, mobile, real-time video. Export → ONNX/TensorRT/CoreML ได้ในบรรทัดเดียว
- **DETR (HuggingFace)** — เหมาะเมื่อต้องการสถาปัตยกรรมเรียบง่าย (ไม่ใช้ NMS) หรือต้องการต่อยอดด้วย transformer-based ecosystem (เช่น integrate กับ vision-language models)
- **Zero-shot (OWLv2 / Grounding DINO)** — เหมาะที่สุดเมื่อยังไม่มี labeled data, prototype เร็ว, หรือต้องตรวจ class ที่หลากหลายและเปลี่ยนบ่อย

### ขั้นตอนต่อไป

- [**YOLO-World**](https://github.com/AILab-CVC/YOLO-World) — open-vocabulary YOLO รวมความเร็วของ YOLO กับ flexibility ของ zero-shot
- [**DINOv2**](https://github.com/facebookresearch/dinov2) — self-supervised vision backbone ที่ใช้กับ detection head ได้
- [**SAM 2**](https://github.com/facebookresearch/segment-anything-2) — ถ้าต้องการ instance segmentation แทน bounding box
- **Fine-tune Grounding DINO** — รวมข้อดีของ open-vocab กับ accuracy ที่สูงขึ้น

### Reference

- [Ultralytics YOLO11 docs](https://docs.ultralytics.com/)
- [HuggingFace Object Detection guide](https://huggingface.co/docs/transformers/tasks/object_detection)
- [Carion et al., DETR (2020)](https://arxiv.org/abs/2005.12872)
- [Minderer et al., OWLv2 (2023)](https://arxiv.org/abs/2306.09683)
- [Liu et al., Grounding DINO (2023)](https://arxiv.org/abs/2303.05499)
- [Pothole dataset by Atikur Rahman Chitholian](https://www.kaggle.com/datasets/chitholian/annotated-potholes-dataset)

---

🎉 **จบ notebook** — ขอบคุณที่อ่านจนจบ! ถ้าโจทย์ของคุณคล้ายกับ pothole detection (single class, dataset เล็ก ๆ, จำเป็นต้อง deploy บน edge) ลองใช้ Path A (YOLO) เป็นจุดเริ่มต้น
